# SWAN Model Runner: Setup, Execution & Output Parsing

---

## Overview

The **SWAN** (Simulating WAves Nearshore) model is a third-generation spectral wave model developed at Delft University of Technology. It solves the spectral action balance equation to compute wave parameters in coastal waters, accounting for:

- Wind generation
- Whitecapping dissipation
- Bottom friction
- Depth-induced breaking
- Refraction and shoaling
- Nonlinear wave-wave interactions

This notebook provides a **Python interface** for:

1. **Modifying SWAN input files** to inject custom output commands
2. **Invoking the SWAN executable** via `subprocess`
3. **Parsing SWAN output files** (TABLE text and BLOCK `.MAT` binary formats)

These utilities are used by the coupled site-selection and optimization notebooks later in the workflow.

### Prerequisites

- SWAN must be compiled and available on your system PATH (see the repository README for installation instructions)
- Dependencies: `numpy`, `scipy` (for `.MAT` files)

### SWAN Input File Structure

A SWAN input file (`INPUT`) is a sequence of keyword commands:

```
PROJECT 'name' ...
CGRID ...          ! Computational grid
INPgrid BOTtom ... ! Bathymetry input specification
READinp BOTtom ... ! Read bathymetry file
BOUnd ...          ! Boundary conditions
GEN3 ...           ! Physics: wave generation
...
COMPUTE ...        ! Run the simulation
STOP
```

Our runner inserts output commands (TABLE, BLOCK, POINTS) **just before the COMPUTE line**.

In [1]:
import os
import re
import shutil
import subprocess
import numpy as np

## 1. Prepare SWAN Input File

The `prepare_input` function takes an existing SWAN input template, copies it, and inserts additional commands (e.g., output requests) immediately before the `COMPUTE` line. This allows us to programmatically request specific outputs without manually editing the input file each time.

**How it works:**
1. Read all lines from the template
2. Find the line starting with `COMP` (the COMPUTE command)
3. Insert the new commands just before it
4. Write the modified file

In [ ]:
def prepare_input(template_path, output_path, extra_commands):
    """
    Copy a SWAN input template and insert extra commands before COMPUTE.

    Parameters
    ----------
    template_path : str  -- path to the original INPUT file.
    output_path   : str  -- path for the modified copy.
    extra_commands: list[str] -- SWAN commands to insert.
    """
    with open(template_path) as f:
        lines = f.readlines()

    # Find the COMPUTE line
    idx = next(
        (i for i, l in enumerate(lines) if l.strip().upper().startswith('COMP')),
        None,
    )
    if idx is None:
        raise ValueError("COMPUTE not found in SWAN input file")

    insert = ['!\n', '!--- output added by swan_runner ---\n']
    for cmd in extra_commands:
        insert.append(cmd + '\n')
    insert.append('!\n')

    lines = lines[:idx] + insert + lines[idx:]
    with open(output_path, 'w') as f:
        f.writelines(lines)
    print(f"Prepared: {output_path}")
    return output_path

## 2. Write Points File

SWAN can output results at specific geographic locations defined in a **points file**. Each line contains a longitude-latitude pair. This is used by the AHP-TOPSIS site-selection workflow to extract wave parameters at candidate deployment sites.

In [ ]:
def write_points_file(filepath, locations):
    """Write a SWAN-format points file (lon lat per line)."""
    with open(filepath, 'w') as f:
        for lon, lat in locations:
            f.write(f'{lon:.6f} {lat:.6f}\n')
    print(f"Wrote {len(locations)} points to {filepath}")

## 3. Execute SWAN

The `run_swan` function invokes the SWAN executable as a subprocess. Key parameters:

| Parameter | Description |
|-----------|-------------|
| `model_dir` | Working directory containing all SWAN files (INPUT, bathymetry, boundary files, etc.) |
| `input_name` | Name of the input file (without extension) |
| `swan_cmd` | The SWAN executable command (`swanrun`, `swan.exe`, etc.) |
| `nproc` | Number of OpenMP threads for parallel execution |
| `timeout` | Maximum runtime in seconds before the process is killed |

After execution, SWAN writes a `norm_end` file indicating normal completion. If this file is missing and the return code is non-zero, the function raises an error.

In [ ]:
def run_swan(model_dir, input_name='INPUT', swan_cmd='swanrun', nproc=4,
             timeout=7200):
    """
    Execute SWAN in *model_dir*.

    Returns
    -------
    subprocess.CompletedProcess
    """
    cmd = f'{swan_cmd} -input {input_name}'
    if nproc > 1:
        cmd += f' -omp {nproc}'

    print(f"    $ {cmd}   (in {model_dir})")
    result = subprocess.run(
        cmd, shell=True, cwd=model_dir,
        capture_output=True, text=True, timeout=timeout,
    )

    norm_end = os.path.join(model_dir, 'norm_end')
    if os.path.exists(norm_end):
        with open(norm_end) as f:
            print(f"    SWAN: {f.read().strip()}")
    elif result.returncode != 0:
        raise RuntimeError(f"SWAN failed (rc={result.returncode}):\n{result.stderr}")

    return result

## 4. Output Readers

SWAN produces two main output formats:

### 4.1 TABLE Output

A text file with `%`-prefixed header lines followed by space-delimited numerical data. Each row corresponds to a point at a timestep. The variable names (e.g., `Hsig`, `RTpeak`, `Dir`, `Depth`) are extracted from the header.

In [ ]:
def read_table(filepath):
    """
    Parse a SWAN TABLE output file.

    Returns
    -------
    dict -- {variable_name: np.array} for each column.
    """
    headers, data_lines = [], []
    with open(filepath) as f:
        for line in f:
            s = line.strip()
            if s.startswith('%'):
                headers.append(s)
            elif s:
                data_lines.append(s)

    # Variable names are in the 2nd-to-last header line
    var_names = headers[-2].lstrip('%').split()
    data = np.loadtxt(data_lines)
    if data.ndim == 1:
        data = data.reshape(1, -1)

    return {v: data[:, i] for i, v in enumerate(var_names) if i < data.shape[1]}

### 4.2 BLOCK Output (`.MAT` format)

SWAN can write 2-D gridded output in MATLAB `.MAT` format (via `LAY-OUT 3`). Each variable at each timestep is stored as a separate matrix with a name like `Hsig_20240101_030000`. We use `scipy.io.loadmat` to read these files and a helper function to stack all timesteps into a 3-D array `(time, ny, nx)`.

In [ ]:
def read_mat_block(filepath):
    """
    Read a SWAN .MAT BLOCK output file.

    Returns
    -------
    dict from scipy.io.loadmat (keys = variable names like 'Hsig_YYYYMMDD_HHMMSS').
    """
    import scipy.io
    return scipy.io.loadmat(filepath)


def extract_block_variable(mat_data, prefix):
    """
    Collect all timestep arrays for a variable prefix (e.g. 'Hsig')
    and return a 3-D array (time, ny, nx).
    """
    pattern = re.compile(rf'^{prefix}_\d{{8}}_\d{{6}}$')
    keys = sorted(k for k in mat_data if pattern.match(k))
    if not keys:
        return None
    return np.stack([mat_data[k] for k in keys], axis=0)

## 5. Utility: Check SWAN Availability

Before attempting to run SWAN, we can check whether the executable is found on the system PATH.

In [ ]:
def check_swan_available(swan_cmd='swanrun'):
    """Return True if SWAN executable is found on PATH."""
    return shutil.which(swan_cmd) is not None

# Quick check
if check_swan_available():
    print("SWAN is available on your system.")
else:
    print("WARNING: 'swanrun' not found on PATH.")
    print("The coupled notebooks (site selection, GA optimization) require SWAN.")
    print("See the repository README for installation instructions.")

## 6. Example: Inspect a SWAN Input File

Let's read the California Bay SWAN input file to see its structure. This helps understand what grid, physics, and boundary settings are configured.

In [ ]:
# Path to the California Bay SWAN input
input_path = os.path.join(os.path.dirname(os.getcwd()), '..', 'california_bay.F', 'INPUT')

if os.path.exists(input_path):
    with open(input_path) as f:
        content = f.read()
    print("=" * 60)
    print("California Bay SWAN Input File")
    print("=" * 60)
    print(content)
else:
    print(f"Input file not found at: {input_path}")

## Summary

This notebook defined the core SWAN interface functions:

| Function | Purpose |
|----------|--------|
| `prepare_input()` | Inject output commands into a SWAN input template |
| `write_points_file()` | Create a SWAN-format points file for output locations |
| `run_swan()` | Execute the SWAN model as a subprocess |
| `read_table()` | Parse SWAN TABLE text output |
| `read_mat_block()` | Read SWAN BLOCK `.MAT` binary output |
| `extract_block_variable()` | Stack timestep matrices into a 3-D array |
| `check_swan_available()` | Verify SWAN is installed |

These functions are imported by the downstream notebooks:
- `03_site_selection/swan_coupled_ahp_topsis.ipynb`
- `04_optimization/swan_coupled_ga_array.ipynb`